In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import GroupShuffleSplit
import ast

In [2]:
# 1. Cargar datos
path_data = '/Users/monicaromero/PycharmProjects/afasia_cat/notebooks_202412/data/'
df = pd.read_csv(path_data + 'df_transcrip_audio_metrics.csv', encoding='utf-8')

In [3]:
def convert_to_list(value):
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return None

# Procesar la columna 'bert_embedding'
df['bert_embedding'] = df['bert_embedding'].apply(convert_to_list)

# Eliminar filas con valores no válidos en 'bert_embedding'
df = df[df['bert_embedding'].notnull()]

# Expandir 'bert_embedding' en columnas individuales
bert_columns = [f'bert_{i}' for i in range(len(df['bert_embedding'].iloc[0]))]
bert_df = pd.DataFrame(df['bert_embedding'].tolist(), columns=bert_columns)

# Concatenar las nuevas columnas con el DataFrame original (sin la columna original)
df = pd.concat([df.drop(columns=['bert_embedding']), bert_df], axis=1)

# Seleccionar características (X) y la variable objetivo (y)
irrelevant_columns = ['Inicio', 'Fin', 'Transcrip_name', 'name_chunk_audio', 'name_chunk_audio_path', 'NumId']
X = df.drop(columns=irrelevant_columns + ['Grup'])  # Reemplaza 'Grup' por tu variable objetivo
X = X.select_dtypes(include=['number'])  # Asegurarse de que solo queden columnas numéricas
y = df['Grup']

In [4]:
# Dividir los datos respetando los grupos
groups = df["CIP"]  # Columna que identifica a cada paciente
gss = GroupShuffleSplit(test_size=0.3, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

# Crear los conjuntos de entrenamiento y prueba
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [5]:
df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()

In [6]:
print("Tamaño del conjunto de entrenamiento:", df_train.shape)
print("Tamaño del conjunto de test:", df_test.shape)

Tamaño del conjunto de entrenamiento: (254, 905)
Tamaño del conjunto de test: (289, 905)


In [7]:
print("Pacientes en entrenamiento:", df_train["CIP"].unique())
print("Pacientes en test:", df_test["CIP"].unique())

Pacientes en entrenamiento: ['02_008' '02_003' '01_005' '01_007' '01_001' '01_004']
Pacientes en test: ['02_007' '02_002' '01_003']


In [8]:
print("Distribución de 'Grup' en entrenamiento:")
print(df_train["Grup"].value_counts())

print("\nDistribución de 'Grup' en test:")
print(df_test["Grup"].value_counts())

Distribución de 'Grup' en entrenamiento:
Grup
1    217
2     37
Name: count, dtype: int64

Distribución de 'Grup' en test:
Grup
2    183
1    106
Name: count, dtype: int64


In [9]:
print("Entrenamiento - Distribución CIP vs Grup:\n", 
      pd.crosstab(df_train["CIP"], df_train["Grup"]))
print("\nTest - Distribución CIP vs Grup:\n", 
      pd.crosstab(df_test["CIP"], df_test["Grup"]))

Entrenamiento - Distribución CIP vs Grup:
 Grup     1   2
CIP           
01_001  30   0
01_004  92   0
01_005  42   0
01_007  49   0
02_003   4   0
02_008   0  37

Test - Distribución CIP vs Grup:
 Grup     1    2
CIP            
01_003  81    0
02_002  25    0
02_007   0  183


# Resumen de los Grupos de Entrenamiento y Test

## Conjunto de Entrenamiento

**Pacientes en entrenamiento:**

| CIP      | Grupo | TipoAfasia             | LLangWAB | Edad | Género | #chunks | QA   |
|----------|-------|------------------------|----------|------|--------|---------|------|
| 01_001   | 1     | Motora (No Fluente)    | Català   | 62   | Hombre | 30      | 44.3 |
| 01_005   | 1     | Motora (No Fluente)    | Català   | 82   | Hombre | 42      | 94.9 |
| 01_007   | 1     | Anómica (Fluente)      | Català   | 52   | Hombre | 49      | 96.7 |
| 02_003   | 1     | Motora (No Fluente)    | Castellà | 69   | Hombre | 4       | 41.2 |
| 02_008   | 1     | Anómica (Fluente)      | Castellà | 71   | Mujer  | 37      | 82.6 |
| 01_004   | 2     | Motora (No Fluente)    | Català   | 68   | Hombre | 92      | 87.7 |

**Estadísticas generales (Entrenamiento):**
- **Número total de pacientes:** 6
- **Total de chunks:** 254
- **Distribución de Tipo de Afasia:**
  - Motora (No Fluente): 4 pacientes
  - Anómica (Fluente): 2 pacientes
- **Grupos:** 5 pacientes en Grupo=1, 1 paciente en Grupo=2
- **Idiomas (LLengWAB):** 4 Català, 2 Castellà
- **Promedio de edad:** 67.33 años
- **Género:** 5 hombres y 1 mujer

---

## Conjunto de Test

**Pacientes en test:**

| CIP      | Grupo | TipoAfasia                 | LLangWAB | Edad | Género | #chunks | QA   |
|----------|-------|----------------------------|----------|------|--------|---------|------|
| 01_003   | 2     | Motora Transcortical (No Fluente) | Català   | 67   | Hombre | 81      | 34.7 |
| 02_002   | 2     | Anómica (Fluente)          | Castellà | 83   | Hombre | 25      | 46.8 |
| 02_007   | 2     | Motora Transcortical (No Fluente) | Castellà | 53   | Mujer  | 183     | 41.8 |

**Estadísticas generales (Test):**
- **Número total de pacientes:** 3
- **Total de chunks:** 289
- **Distribución de Tipo de Afasia:**
  - Motora Transcortical (No Fluente): 2 pacientes
  - Anómica (Fluente): 1 paciente
- **Grupos:** todos en Grupo=2
- **Idiomas (LLengWAB):** 1 Català, 2 Castellà
- **Promedio de edad:** 67.67 años
- **Género:** 2 hombres y 1 mujer

---

## Comentarios Finales

- La partición agrupa a ciertos pacientes enteros en **train** y a otros en **test**. Esto explica por qué el test tiene más *chunks* totales debido a que `02_007` aporta 183 *chunks* solo él.
- La distribución de tipos de afasia, género, grupo e idioma se ve bastante **desbalanceada**:
  - En **train**, predominan los hombres (5 de 6) y el grupo=1.
  - En **test**, todos los pacientes pertenecen al grupo=2.
- **Recomendaciones:**
  - Considerar otras estrategias de partición como **LeaveOneGroupOut** o **GroupKFold**, especialmente cuando hay pocos pacientes.
  - Re-equilibrar las clases o ajustar los pesos de clases para compensar las diferencias durante el entrenamiento.

In [10]:
# Normalizar las características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Modelo base: Regresión logística
log_reg = LogisticRegression(random_state=42, max_iter=1000, penalty='l2', solver='lbfgs', class_weight='balanced')

# Validación cruzada
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(log_reg, X_train_scaled, y_train, cv=cv, scoring='accuracy')
print(f"Accuracy promedio de la regresión logística (CV): {scores.mean():.2f}")

# Entrenar modelo base
log_reg.fit(X_train_scaled, y_train)

# Evaluar modelo base
y_pred = log_reg.predict(X_test_scaled)
print("Reporte de clasificación para el modelo base:")
print(classification_report(y_test, y_pred))

# Ensamblaje multimodal (early fusion)
ensemble_clf = VotingClassifier(
    estimators=[
        ('log_reg', log_reg),
    ],
    voting='soft'  # Promediar probabilidades
)

# Entrenar modelo ensamblado
ensemble_clf.fit(X_train_scaled, y_train)

# Evaluar modelo ensamblado
y_pred_ensemble = ensemble_clf.predict(X_test_scaled)
print("Reporte de clasificación (modelo ensamblado):")
print(classification_report(y_test, y_pred_ensemble))

Accuracy promedio de la regresión logística (CV): 0.99
Reporte de clasificación para el modelo base:
              precision    recall  f1-score   support

           1       0.48      0.89      0.63       106
           2       0.87      0.45      0.60       183

    accuracy                           0.61       289
   macro avg       0.68      0.67      0.61       289
weighted avg       0.73      0.61      0.61       289

Reporte de clasificación (modelo ensamblado):
              precision    recall  f1-score   support

           1       0.48      0.89      0.63       106
           2       0.87      0.45      0.60       183

    accuracy                           0.61       289
   macro avg       0.68      0.67      0.61       289
weighted avg       0.73      0.61      0.61       289



In [12]:
# df_results.to_csv("resultados_clasificacion.csv", index=False)